In [1]:
!ls /kaggle/input/datasets/kamikazeeeeeee88/
!ls /kaggle/input/datasets/kamikazeeeeeee88/nps-drone-train-part1 
!ls /kaggle/input/datasets/kamikazeeeeeee88/nps-drone-train-part1/images_train| head -n 5
!ls /kaggle/input/datasets/kamikazeeeeeee88/nps-drone-train-part1/labels_train| head -n 5
!cat /kaggle/input/datasets/kamikazeeeeeee88/nps-drone-train-part1/labels_train/Clip_001_00056.txt




nps-drone-test	nps-drone-train-part1  nps-drone-train-part2  nps-drone-val
images_train  labels_train
Clip_001_00056.png
Clip_001_00057.png
Clip_001_00058.png
Clip_001_00059.png
Clip_001_00060.png
ls: write error: Broken pipe
Clip_001_00056.txt
Clip_001_00057.txt
Clip_001_00058.txt
Clip_001_00059.txt
Clip_001_00060.txt
ls: write error: Broken pipe
0 0.560937 0.184259 0.005208 0.009259


In [2]:
# =============================================================================
# CELL 1 — Install dependencies
# =============================================================================
import subprocess, sys

pkgs = [
    "wandb",
    "timm",
    "albumentations>=1.4.0",
    "torchmetrics",
    "pycocotools",
    "einops",
    "tqdm",
    "matplotlib",
    "seaborn",
    "pandas",
    "numpy",
    "opencv-python-headless",
    "scipy",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.4 MB/s eta 0:00:00
All packages installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [3]:
# =============================================================================
# CELL 2 — Imports
# =============================================================================
import os, json, math, time, random, glob, copy, warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import maximum_filter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.ops as tv_ops

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

import wandb
from kaggle_secrets import UserSecretsClient

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True
print("Imports done. Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Imports done. Torch: 2.10.0+cu128 | CUDA: True


In [4]:
# =============================================================================
# CELL 3 — Global configuration (single source of truth)
# =============================================================================
CFG = {
    # ── project ──────────────────────────────────────────────────────────────
    "project"        : "drone-detection-nps",
    "run_name"       : "heatmap-guided-sparse-detector-v1",
    "seed"           : 42,

    # ── data ─────────────────────────────────────────────────────────────────
    "data_root"      : "/kaggle/input/datasets/kamikazeeeeeee88",
    "train_parts"    : ["nps-drone-train-part1", "nps-drone-train-part2"],
    "val_dir"        : "nps-drone-val",
    "test_dir"       : "nps-drone-test",
    "img_size"       : 640,
    "temporal_len"   : 3,          # number of consecutive frames fed to model

    # ── heatmap ──────────────────────────────────────────────────────────────
    "heatmap_stride" : 4,          # output stride of BiFPN → heatmap H/4 × W/4
    "heatmap_sigma"  : 2.0,        # Gaussian sigma for center targets
    "topk"           : 50,         # candidate peaks kept per image

    # ── backbone ─────────────────────────────────────────────────────────────
    "backbone"       : "efficientnet_b0",
    "pretrained"     : True,
    "backbone_out_channels": [40, 112, 320],  # C3/C4/C5 for EB0

    # ── BiFPN ────────────────────────────────────────────────────────────────
    "bifpn_channels" : 64,
    "bifpn_layers"   : 3,

    # ── ROIAlign ─────────────────────────────────────────────────────────────
    "roi_output_size": 7,
    "roi_levels"     : [0, 1, 2],  # P2, P3, P4 index into FPN list

    # ── refinement head ──────────────────────────────────────────────────────
    "refine_hidden"  : 256,
    "num_classes"    : 1,          # drone only

    # ── training ─────────────────────────────────────────────────────────────
    "epochs"         : 60,
    "batch_size"     : 4,          # Kaggle T4: safe with amp
    "accum_steps"    : 4,          # effective batch = 16
    "num_workers"    : 2,
    "lr"             : 2e-4,
    "weight_decay"   : 1e-4,
    "warmup_epochs"  : 5,
    "cosine_min_lr"  : 1e-6,
    "clip_grad"      : 2.0,

    # ── loss weights ─────────────────────────────────────────────────────────
    "heatmap_loss_w" : 1.0,
    "obj_loss_w"     : 1.0,
    "box_loss_w"     : 2.0,

    # ── focal loss params ────────────────────────────────────────────────────
    "focal_alpha"    : 2.0,
    "focal_beta"     : 4.0,

    # ── NMS ──────────────────────────────────────────────────────────────────
    "nms_iou_thr"    : 0.3,
    "conf_thr"       : 0.3,

    # ── misc ─────────────────────────────────────────────────────────────────
    "save_dir"       : "/kaggle/working/checkpoints",
    "amp"            : True,
}

os.makedirs(CFG["save_dir"], exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Config set.")

Device: cuda
Config set.


In [5]:
# =============================================================================
# CELL 4 — W&B initialisation
# =============================================================================
secrets   = UserSecretsClient()
WANDB_KEY = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_KEY)

run = wandb.init(
    project   = CFG["project"],
    name      = CFG["run_name"],
    config    = CFG,
    save_code = True,
)
print(f"W&B run: {run.url}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bscs22030 (bscs22030-information-technology-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/bscs22030-information-technology-university/drone-detection-nps/runs/5bq6hnnc


In [6]:
# =============================================================================
# CELL 5 — Dataset preparation: build file-lists and COCO JSON for val/test
# =============================================================================

def yolo_to_abs(cx, cy, w, h, W, H):
    """Convert YOLO normalised [cx cy w h] → absolute [x1 y1 x2 y2]."""
    cx, cy, w, h = cx * W, cy * H, w * W, h * H
    return cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2


def load_clip_index(image_dir, label_dir):
    """
    Group images by clip prefix and return sorted list of
    (img_path, label_path) tuples.  Handles missing labels gracefully.
    """
    img_paths = sorted(glob.glob(str(image_dir / "*.png")) +
                       glob.glob(str(image_dir / "*.jpg")))
    pairs = []
    for ip in img_paths:
        stem = Path(ip).stem
        lp   = label_dir / f"{stem}.txt"
        pairs.append((ip, str(lp) if lp.exists() else None))
    return pairs


def build_temporal_triplets(pairs, T=3):
    """
    Group consecutive frames into overlapping windows of length T.
    Returns list of [(img_path, lbl_path), …] of length T,
    each window sharing the same clip (same Clip_XXX prefix).
    """
    # Group by clip name
    clips = defaultdict(list)
    for ip, lp in pairs:
        clip_id = "_".join(Path(ip).stem.split("_")[:2])   # e.g. Clip_001
        clips[clip_id].append((ip, lp))

    triplets = []
    for clip_id in sorted(clips.keys()):
        frames = clips[clip_id]
        for i in range(len(frames) - T + 1):
            triplets.append(frames[i : i + T])
    return triplets


def build_dataset_splits():
    root = Path(CFG["data_root"])
    train_triplets = []
    for part in CFG["train_parts"]:
        img_dir = root / part / "images_train"
        lbl_dir = root / part / "labels_train"
        pairs   = load_clip_index(img_dir, lbl_dir)
        train_triplets.extend(build_temporal_triplets(pairs, CFG["temporal_len"]))

    val_img = root / CFG["val_dir"] / "images_val"
    val_lbl = root / CFG["val_dir"] / "labels_val"
    val_triplets = build_temporal_triplets(
        load_clip_index(val_img, val_lbl), CFG["temporal_len"])

    test_img = root / CFG["test_dir"] / "images_test"
    test_lbl = root / CFG["test_dir"] / "labels_test"
    test_triplets = build_temporal_triplets(
        load_clip_index(test_img, test_lbl), CFG["temporal_len"])

    print(f"Train triplets : {len(train_triplets)}")
    print(f"Val   triplets : {len(val_triplets)}")
    print(f"Test  triplets : {len(test_triplets)}")
    return train_triplets, val_triplets, test_triplets


def read_labels(lbl_path):
    """Return list of [cx, cy, w, h] in YOLO normalised coords."""
    if lbl_path is None or not os.path.exists(lbl_path):
        return []
    boxes = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                boxes.append([float(x) for x in parts[1:5]])
    return boxes


def build_coco_json(triplets, out_path, img_size=640):
    """
    GT boxes recorded in the SAME 640x640 letterboxed coordinate space
    that the model operates in (LongestMaxSize + centered PadIfNeeded).
    """
    images, annotations = [], []
    ann_id = 1
    for img_id, triplet in enumerate(triplets):
        img_path, lbl_path = triplet[-1]
        img = cv2.imread(img_path)
        H, W = img.shape[:2]

        scale    = img_size / max(H, W)
        new_H    = int(round(H * scale))
        new_W    = int(round(W * scale))
        pad_top  = (img_size - new_H) // 2
        pad_left = (img_size - new_W) // 2

        images.append({"id": img_id, "file_name": img_path,
                        "width": img_size, "height": img_size})

        for cx, cy, bw, bh in read_labels(lbl_path):
            x1, y1, x2, y2 = yolo_to_abs(cx, cy, bw, bh, W, H)
            x1 = x1 * scale + pad_left
            y1 = y1 * scale + pad_top
            x2 = x2 * scale + pad_left
            y2 = y2 * scale + pad_top

            annotations.append({
                "id"         : ann_id,
                "image_id"   : img_id,
                "category_id": 1,
                "bbox"       : [x1, y1, x2 - x1, y2 - y1],
                "area"       : (x2 - x1) * (y2 - y1),
                "iscrowd"    : 0,
            })
            ann_id += 1

    coco_dict = {
        "images"     : images,
        "annotations": annotations,
        "categories" : [{"id": 1, "name": "drone"}],
    }
    with open(out_path, "w") as f:
        json.dump(coco_dict, f)
    print(f"Saved COCO JSON → {out_path}  ({len(images)} imgs, {len(annotations)} anns)")
    return out_path



train_triplets, val_triplets, test_triplets = build_dataset_splits()

VAL_COCO_JSON  = "/kaggle/working/val_coco.json"
TEST_COCO_JSON = "/kaggle/working/test_coco.json"
build_coco_json(val_triplets,  VAL_COCO_JSON)
build_coco_json(test_triplets, TEST_COCO_JSON)

Train triplets : 32148
Val   triplets : 3745
Test  triplets : 12335
Saved COCO JSON → /kaggle/working/val_coco.json  (3745 imgs, 4648 anns)
Saved COCO JSON → /kaggle/working/test_coco.json  (12335 imgs, 16360 anns)


'/kaggle/working/test_coco.json'

In [7]:
# =============================================================================
# CELL 6 — Augmentation pipelines (temporally consistent)
# =============================================================================

PIXEL_TRAIN = A.Compose([
    A.OneOf([
        A.GaussNoise(var_limit=(10, 50)),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5)),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1)),
    ], p=0.6),
    A.OneOf([
        A.MotionBlur(blur_limit=5),
        A.GaussianBlur(blur_limit=(3, 5)),
    ], p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20,
                         val_shift_limit=20, p=0.4),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.2),
    A.ToFloat(max_value=255.0),
])

PIXEL_VAL = A.Compose([A.ToFloat(max_value=255.0)])

SPATIAL_TRAIN = A.Compose([
    A.LongestMaxSize(max_size=CFG["img_size"]),
    A.PadIfNeeded(min_height=CFG["img_size"], min_width=CFG["img_size"],
                  border_mode=cv2.BORDER_CONSTANT, value=0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.2,
                       rotate_limit=15, border_mode=cv2.BORDER_CONSTANT,
                       value=0, p=0.6),
    A.RandomCrop(height=CFG["img_size"], width=CFG["img_size"], p=0.3),
    A.PadIfNeeded(min_height=CFG["img_size"], min_width=CFG["img_size"],
                  border_mode=cv2.BORDER_CONSTANT, value=0),
],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.3,
    )
)

SPATIAL_VAL = A.Compose([
    A.LongestMaxSize(max_size=CFG["img_size"]),
    A.PadIfNeeded(min_height=CFG["img_size"], min_width=CFG["img_size"],
                  border_mode=cv2.BORDER_CONSTANT, value=0),
],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.3,
    )
)


def sanitise_yolo_boxes(boxes):
    """
    Clamp YOLO [cx, cy, w, h] boxes so that the implied corners stay in
    [0, 1].  Floating-point noise in labels (e.g. -5e-7) causes albumentations
    to hard-reject the entire batch; clamp before and after transforms.
    """
    clean = []
    for box in boxes:
        cx, cy, bw, bh = box
        # clamp centre + half-dims so corners are in [eps, 1-eps]
        eps = 1e-6
        bw  = float(np.clip(bw, eps, 1.0))
        bh  = float(np.clip(bh, eps, 1.0))
        cx  = float(np.clip(cx, bw / 2, 1.0 - bw / 2))
        cy  = float(np.clip(cy, bh / 2, 1.0 - bh / 2))
        clean.append((cx, cy, bw, bh))
    return clean


def apply_temporal_augmentation(frames_bgr, boxes_list, is_train):
    """
    frames_bgr      : list of H×W×3 uint8 arrays (length T)
    boxes_list      : list of [[cx,cy,w,h], ...] per frame (YOLO normalised)
    Returns:
        frames_tensor : T×3×H×W float32 (ImageNet-normalised)
        anchor_boxes  : [[cx,cy,w,h], ...] for the LAST frame after transforms
    """
    spatial_tf = SPATIAL_TRAIN if is_train else SPATIAL_VAL
    pixel_tf   = PIXEL_TRAIN   if is_train else PIXEL_VAL

    anchor_boxes_raw = sanitise_yolo_boxes(boxes_list[-1])

    # Fix a shared spatial random seed so all T frames see identical geometry
    triplet_seed = random.randint(0, 2**31 - 1)

    processed_frames = []
    final_boxes      = anchor_boxes_raw   # fallback if anchor has no boxes

    for i, frame in enumerate(frames_bgr):
        is_anchor = (i == len(frames_bgr) - 1)
        boxes     = anchor_boxes_raw if is_anchor else []
        labels    = [0] * len(boxes)

        random.seed(triplet_seed)
        np.random.seed(triplet_seed)
        out = spatial_tf(image=frame, bboxes=boxes, class_labels=labels)

        if is_anchor:
            # Clamp output boxes again — ShiftScaleRotate can push edges slightly OOB
            final_boxes = sanitise_yolo_boxes(list(out["bboxes"]))

        out2 = pixel_tf(image=out["image"])
        processed_frames.append(out2["image"])          # float32 H×W×3

    frames_np = np.stack(processed_frames, axis=0)      # T,H,W,3
    frames_t  = torch.from_numpy(frames_np.transpose(0, 3, 1, 2))  # T,3,H,W

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    frames_t = (frames_t - mean) / std

    return frames_t, final_boxes


print("Augmentation pipelines ready.")

Augmentation pipelines ready.


In [8]:
# =============================================================================
# CELL 7 — Dataset and DataLoader
# =============================================================================

def gaussian2D(shape, sigma=1.0):
    """Pre-compute a 2D Gaussian kernel on a grid of given shape."""
    m, n   = [(s - 1.) / 2. for s in shape]
    y, x   = np.ogrid[-m:m+1, -n:n+1]
    kernel = np.exp(-(x * x + y * y) / (2 * sigma * sigma))
    kernel[kernel < np.finfo(kernel.dtype).eps * kernel.max()] = 0
    return kernel


def build_gaussian_heatmap(centers_xy, H, W, sigma=2.0):
    """
    Vectorised Gaussian rendering using a pre-computed kernel.
    The kernel is stamped so its analytical centre pixel is exactly 1.0,
    avoiding the sub-pixel misalignment that caused peak < 1.0.
    centers_xy : list of (x, y) in heatmap-pixel space (float)
    """
    hm = np.zeros((H, W), dtype=np.float32)
    if not centers_xy:
        return hm

    radius = max(int(math.ceil(3 * sigma)), 1)
    diameter = 2 * radius + 1
    kernel   = gaussian2D((diameter, diameter), sigma=sigma)
    # Force the centre to exactly 1.0 (eliminates floating-point peak < 1.0)
    kernel[radius, radius] = 1.0

    for (cx, cy) in centers_xy:
        # Snap centre to nearest pixel (the kernel centre IS the peak)
        ix, iy = int(round(cx)), int(round(cy))

        # Destination region on heatmap
        x0h = max(0,  ix - radius);   x1h = min(W, ix + radius + 1)
        y0h = max(0,  iy - radius);   y1h = min(H, iy + radius + 1)

        # Corresponding source region in kernel
        x0k = x0h - (ix - radius);    x1k = x0k + (x1h - x0h)
        y0k = y0h - (iy - radius);    y1k = y0k + (y1h - y0h)

        if x1h <= x0h or y1h <= y0h:
            continue

        hm[y0h:y1h, x0h:x1h] = np.maximum(
            hm[y0h:y1h, x0h:x1h],
            kernel[y0k:y1k, x0k:x1k])

    return hm


class DroneDataset(Dataset):
    def __init__(self, triplets, is_train=True):
        self.triplets = triplets
        self.is_train = is_train
        # Heatmap is now at stride 4 → 160×160
        self.HM_H = CFG["img_size"] // CFG["heatmap_stride"]   # 160
        self.HM_W = CFG["img_size"] // CFG["heatmap_stride"]   # 160

    def __len__(self):
        return len(self.triplets)

    def _load_frame(self, path):
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((CFG["img_size"], CFG["img_size"], 3), dtype=np.uint8)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        triplet    = self.triplets[idx]
        frames_bgr = [self._load_frame(ip) for ip, _ in triplet]
        boxes_list = [read_labels(lp) for _, lp in triplet]

        frames_t, anchor_boxes = apply_temporal_augmentation(
            frames_bgr, boxes_list, self.is_train)

        # ── heatmap target ────────────────────────────────────────────────
        centers      = []
        gt_boxes_abs = []

        for (cx, cy, bw, bh) in anchor_boxes:
            # Centre in heatmap-pixel space
            hm_x = cx * self.HM_W
            hm_y = cy * self.HM_H
            centers.append((hm_x, hm_y))

            # Absolute xyxy in 640×640 image space
            px   = cx  * CFG["img_size"]
            py   = cy  * CFG["img_size"]
            pw   = bw  * CFG["img_size"]
            ph   = bh  * CFG["img_size"]
            gt_boxes_abs.append([px - pw/2, py - ph/2,
                                  px + pw/2, py + ph/2])

        heatmap   = build_gaussian_heatmap(
            centers, self.HM_H, self.HM_W, CFG["heatmap_sigma"])
        heatmap_t = torch.from_numpy(heatmap).unsqueeze(0)   # 1×160×160

        # ── pad GT to fixed size for batching ────────────────────────────
        max_objs = 64
        gt_arr   = np.zeros((max_objs, 4), dtype=np.float32)
        gt_mask  = np.zeros(max_objs,      dtype=np.float32)
        for i, box in enumerate(gt_boxes_abs[:max_objs]):
            gt_arr[i]  = box
            gt_mask[i] = 1.0

        return {
            "frames"  : frames_t,                         # T×3×640×640
            "heatmap" : heatmap_t,                        # 1×160×160
            "gt_boxes": torch.from_numpy(gt_arr),         # max_objs×4
            "gt_mask" : torch.from_numpy(gt_mask),        # max_objs
            "img_id"  : idx,
        }


def build_dataloaders(train_triplets, val_triplets, test_triplets):
    train_ds = DroneDataset(train_triplets, is_train=True)
    val_ds   = DroneDataset(val_triplets,   is_train=False)
    test_ds  = DroneDataset(test_triplets,  is_train=False)

    train_dl = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  num_workers=CFG["num_workers"],
                          pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"],
                          pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=CFG["num_workers"],
                          pin_memory=True)
    return train_dl, val_dl, test_dl


train_dl, val_dl, test_dl = build_dataloaders(
    train_triplets, val_triplets, test_triplets)
print(f"Train batches : {len(train_dl)}")
print(f"Val   batches : {len(val_dl)}")
print(f"Test  batches : {len(test_dl)}")

Train batches : 8037
Val   batches : 937
Test  batches : 3084


In [9]:
# =============================================================================
# CELL 9 — BiFPN implementation (with stride-4 P1 output)
# =============================================================================

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, kernel, padding=padding,
                             groups=in_ch, bias=False)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn  = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))


class BiFPNNode(nn.Module):
    def __init__(self, ch, n_inputs=2):
        super().__init__()
        self.w   = nn.Parameter(torch.ones(n_inputs, dtype=torch.float32))
        self.eps = 1e-4
        self.conv= DepthwiseSeparableConv(ch, ch)

    def forward(self, feats):
        w = F.relu(self.w)
        w = w / (w.sum() + self.eps)
        fused = sum(w[i] * feats[i] for i in range(len(feats)))
        return self.conv(fused)


class BiFPNLayer(nn.Module):
    def __init__(self, ch, num_levels=3):
        super().__init__()
        self.td_nodes = nn.ModuleList(
            [BiFPNNode(ch, n_inputs=2) for _ in range(num_levels - 1)])
        self.bu_nodes = nn.ModuleList(
            [BiFPNNode(ch, n_inputs=3 if i < num_levels - 2 else 2)
             for i in range(num_levels - 1)])

    def forward(self, features):
        n = len(features)
        td = [None] * n
        td[-1] = features[-1]
        for i in range(n - 2, -1, -1):
            up   = F.interpolate(td[i + 1], size=features[i].shape[-2:],
                                 mode="nearest")
            td[i]= self.td_nodes[i]([features[i], up])
        out = [None] * n
        out[0] = td[0]
        for i in range(1, n):
            dn = F.adaptive_avg_pool2d(out[i - 1], features[i].shape[-2:])
            if i < n - 1:
                out[i] = self.bu_nodes[i - 1]([features[i], td[i], dn])
            else:
                out[i] = self.bu_nodes[i - 1]([features[i], dn])
        return out


class BiFPN(nn.Module):
    """
    Inputs : C3/C4/C5 from backbone (strides 8/16/32)
    Outputs: [P1, P2, P3, P4]
             P1 = stride-4  (160×160 at 640 input)  ← heatmap level
             P2 = stride-8  ( 80×80)
             P3 = stride-16 ( 40×40)
             P4 = stride-32 ( 20×20)
    P1 is produced by upsampling P2 through a learned conv, NOT from the
    backbone, so no extra backbone feature is needed.
    """
    def __init__(self, in_channels_list, out_ch, num_layers=3):
        super().__init__()
        # lateral projections for C3, C4, C5
        self.lateral = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.SiLU(inplace=True))
            for c in in_channels_list            # 3 levels: C3/C4/C5
        ])

        # repeated BiFPN over 3 levels (P2/P3/P4)
        self.layers = nn.ModuleList([
            BiFPNLayer(out_ch, num_levels=3)
            for _ in range(num_layers)
        ])

        # upsample P2→P1 (stride 8 → stride 4)
        self.p1_upsample = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            DepthwiseSeparableConv(out_ch, out_ch),
        )

    def forward(self, c_feats):
        # c_feats = [C3, C4, C5]
        feats = [lat(c) for lat, c in zip(self.lateral, c_feats)]  # P2,P3,P4
        for layer in self.layers:
            feats = layer(feats)          # still [P2, P3, P4]
        p1 = self.p1_upsample(feats[0])  # stride-4 feature
        return [p1] + feats              # [P1, P2, P3, P4]


print("BiFPN defined (with stride-4 P1 output).")

BiFPN defined (with stride-4 P1 output).


In [10]:
# =============================================================================
# CELL 10 — Heatmap head, peak extractor, ROIAlign, refinement head
# =============================================================================

class HeatmapHead(nn.Module):
    def __init__(self, in_ch, num_classes=1):
        super().__init__()
        self.head = nn.Sequential(
            DepthwiseSeparableConv(in_ch, in_ch),
            DepthwiseSeparableConv(in_ch, in_ch),
            nn.Conv2d(in_ch, num_classes, 1),
        )

    def forward(self, p1):
        return self.head(p1)   # B×1×160×160  (stride-4 logits)


def extract_peaks(heatmap_logits, topk=50, kernel=3, conf_thr=0.01):
    B    = heatmap_logits.shape[0]
    heat = torch.sigmoid(heatmap_logits)
    pad  = kernel // 2
    hmax = F.max_pool2d(heat, kernel, stride=1, padding=pad)
    keep = (heat == hmax).float() * heat

    peaks_list = []
    for b in range(B):
        flat = keep[b, 0].reshape(-1)
        k    = min(topk, int((flat > conf_thr).sum().item()))
        k    = max(k, 1)
        vals, inds = flat.topk(k)
        H, W  = keep.shape[-2:]
        ys    = inds // W
        xs    = inds %  W
        coords = torch.stack([ys, xs], dim=1).float()
        peaks_list.append((coords, vals))
    return peaks_list


class RefinementHead(nn.Module):
    def __init__(self, roi_size, num_fpn_levels, fpn_ch, hidden=256):
        super().__init__()
        flat_dim = num_fpn_levels * fpn_ch * roi_size * roi_size
        self.mlp = nn.Sequential(
            nn.Linear(flat_dim, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2),
            nn.SiLU(inplace=True),
        )
        self.obj_head = nn.Linear(hidden // 2, 1)
        self.box_head = nn.Linear(hidden // 2, 4)

    def forward(self, roi_feats):
        h   = self.mlp(roi_feats)
        obj = self.obj_head(h)
        box = self.box_head(h)
        return obj, box


print("Heads defined.")

Heads defined.


In [11]:
# =============================================================================
# CELL 11 — Full HeatmapGuidedDetector model
# =============================================================================

class TemporalFusion(nn.Module):
    def __init__(self, ch, T):
        super().__init__()
        self.T    = T
        self.fuse = nn.Sequential(
            nn.Conv2d(ch * T, ch, 1, bias=False),
            nn.BatchNorm2d(ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        BT, C, H, W = x.shape
        B = BT // self.T
        return self.fuse(x.view(B, self.T * C, H, W))


class HeatmapGuidedDetector(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        T        = cfg["temporal_len"]
        fpn_ch   = cfg["bifpn_channels"]
        roi_size = cfg["roi_output_size"]

        # ── backbone ──────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            cfg["backbone"],
            pretrained=cfg["pretrained"],
            features_only=True,
            out_indices=(2, 3, 4),     # C3/C4/C5 at stride 8/16/32
        )
        bb_ch = cfg["backbone_out_channels"]   # [40, 112, 320] for EB0

        # ── temporal fusion (one per backbone level) ──────────────────────
        self.temporal_fuse = nn.ModuleList(
            [TemporalFusion(c, T) for c in bb_ch])

        # ── BiFPN → [P1(s4), P2(s8), P3(s16), P4(s32)] ──────────────────
        self.bifpn = BiFPN(bb_ch, fpn_ch, cfg["bifpn_layers"])

        # ── heatmap head on P1 (stride-4, 160×160) ────────────────────────
        self.hm_head = HeatmapHead(fpn_ch, cfg["num_classes"])

        # ── ROI levels: use P1/P2/P3 (indices 0,1,2 into BiFPN output) ───
        # These map to strides 4/8/16 — appropriate for ~4–20px drones
        roi_levels   = [0, 1, 2]
        n_roi_levels = len(roi_levels)
        self.roi_levels = roi_levels

        # ── refinement head ───────────────────────────────────────────────
        self.refine_head = RefinementHead(
            roi_size, n_roi_levels, fpn_ch, cfg["refine_hidden"])

        self.roi_size = roi_size
        self.fpn_ch   = fpn_ch
        # Actual output stride of heatmap = 4
        self.hm_stride  = 4
        self.topk       = cfg["topk"]
        self.img_size   = cfg["img_size"]

    # ─────────────────────────────────────────────────────────────────────
    def _extract_backbone(self, frames):
        B, T, C, H, W = frames.shape
        x     = frames.view(B * T, C, H, W)
        feats = self.backbone(x)
        fused = [self.temporal_fuse[i](feats[i]) for i in range(3)]
        return fused   # [C3(s8), C4(s16), C5(s32)]  each B×C×h×w

    # ─────────────────────────────────────────────────────────────────────
    def _roi_align_multi(self, fpn_feats, peaks_batch, B):
        all_roi_feats = []
        all_anchors   = []
        batch_splits  = []

        for b in range(B):
            coords, scores = peaks_batch[b]   # N×2 (y,x in heatmap)
            N = coords.shape[0]
            batch_splits.append(N)

            # heatmap is at stride self.hm_stride → multiply to get image px
            cxs = coords[:, 1] * self.hm_stride
            cys = coords[:, 0] * self.hm_stride

            level_roi_list = []
            for lvl_idx in self.roi_levels:
                fmap     = fpn_feats[lvl_idx]          # B×C×h×w
                h_f, w_f = fmap.shape[-2:]
                scale_x  = w_f / self.img_size
                scale_y  = h_f / self.img_size

                # ROI centres in feature-map pixels
                fc_x = cxs * scale_x    # centre x on feature map
                fc_y = cys * scale_y    # centre y on feature map
                half = self.roi_size / 2.0

                x1_f = fc_x - half
                y1_f = fc_y - half
                x2_f = fc_x + half
                y2_f = fc_y + half

                batch_col = torch.full(
                    (N, 1), float(b), device=fmap.device)
                rois_in = torch.stack(
                    [batch_col[:, 0], x1_f, y1_f, x2_f, y2_f], dim=1)

                roi_out = tv_ops.roi_align(
                    fmap, rois_in,
                    output_size=self.roi_size,
                    spatial_scale=1.0,
                    sampling_ratio=2,
                    aligned=True)               # N×C×roi×roi
                level_roi_list.append(roi_out)

            cat_roi = torch.cat(level_roi_list, dim=1)   # N×(C*3)×roi×roi
            all_roi_feats.append(cat_roi.view(N, -1))

            anchors_b = torch.stack([
                cxs, cys,
                torch.full_like(cxs, 8.0),   # was 16.0 → now matches measured ~7.5-8px GT
                torch.full_like(cys, 8.0),
            ], dim=1)                                  # N×4
            all_anchors.append(anchors_b)

        all_roi_feats_cat = torch.cat(all_roi_feats, dim=0)
        return all_roi_feats_cat, all_anchors, batch_splits

    # ─────────────────────────────────────────────────────────────────────
    def forward(self, frames):
        B         = frames.shape[0]
        c_feats   = self._extract_backbone(frames)         # [C3,C4,C5]
        fpn_feats = self.bifpn(c_feats)                    # [P1,P2,P3,P4]

        # Heatmap from P1 (stride-4 → 160×160)
        hm_logits = self.hm_head(fpn_feats[0])             # B×1×160×160

        with torch.no_grad():
            peaks_batch = extract_peaks(
                hm_logits.detach(), topk=self.topk, conf_thr=0.01)

        roi_feats, anchors, splits = self._roi_align_multi(
            fpn_feats, peaks_batch, B)

        obj_logits, box_deltas = self.refine_head(roi_feats)
        return hm_logits, obj_logits, box_deltas, anchors, splits


# ── rebuild model with corrected architecture ────────────────────────────
model = HeatmapGuidedDetector(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {n_params/1e6:.2f}M")

# Quick shape sanity check
with torch.no_grad():
    dummy = torch.randn(2, CFG["temporal_len"], 3,
                        CFG["img_size"], CFG["img_size"]).to(DEVICE)
    hm, obj, box, anc, spl = model(dummy)
    print(f"hm_logits : {hm.shape}   ← must be [B,1,160,160]")
    print(f"obj_logits: {obj.shape}")
    print(f"box_deltas: {box.shape}")
    assert hm.shape[-2:] == (160, 160), "Stride mismatch — check BiFPN"
    print("Shape check PASSED.")


# ── CenterNet-style bias init for heatmap output conv ────────────────────
# Prevents sigmoid(hm) ≈ 0.5 at init, which causes 102400 negative pixels
# all contributing full loss before any learning happens.
prior_prob = 0.01
bias_init  = -math.log((1.0 - prior_prob) / prior_prob)   # = -4.595

for m in model.hm_head.head.modules():
    if isinstance(m, nn.Conv2d):
        last_hm_conv = m
nn.init.constant_(last_hm_conv.bias, bias_init)
nn.init.normal_(last_hm_conv.weight, std=0.001)
print(f"Heatmap conv bias → {bias_init:.3f}  (sigmoid ≈ {prior_prob})")

# ── verify before training ────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    dummy  = torch.randn(1, CFG["temporal_len"], 3,
                         CFG["img_size"], CFG["img_size"]).to(DEVICE)
    hm_t, *_ = model(dummy)
    sig_mean = torch.sigmoid(hm_t).mean().item()
    n_hot    = (torch.sigmoid(hm_t) > 0.5).sum().item()
print(f"After init: sigmoid(hm) mean={sig_mean:.4f}  hot_pixels={n_hot}  "
      f"(want mean≈0.01, hot_pixels≈0)")
model.train()

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Model params: 6.49M
hm_logits : torch.Size([2, 1, 160, 160])   ← must be [B,1,160,160]
obj_logits: torch.Size([100, 1])
box_deltas: torch.Size([100, 4])
Shape check PASSED.
Heatmap conv bias → -4.595  (sigmoid ≈ 0.01)
After init: sigmoid(hm) mean=0.0100  hot_pixels=0  (want mean≈0.01, hot_pixels≈0)


HeatmapGuidedDetector(
  (backbone): EfficientNetFeatures(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
    

In [12]:
# =============================================================================
# CELL 12 — Loss functions (focal loss correctly normalised)
# =============================================================================

class FocalLossHeatmap(nn.Module):
    """
    CenterNet focal loss with threshold-based positive masking.
    pos_mask uses >= 0.99 instead of == 1.0 because floating-point
    Gaussian rendering never reaches exactly 1.0 in practice.
    """
    def __init__(self, alpha=2.0, beta=4.0, pos_thresh=0.99):
        super().__init__()
        self.alpha      = alpha
        self.beta       = beta
        self.pos_thresh = pos_thresh

    def forward(self, pred_logits, target):
        pred = torch.sigmoid(pred_logits).clamp(1e-6, 1 - 1e-6)
        tgt  = target.clamp(0.0, 1.0)

        pos_mask = tgt >= self.pos_thresh   # ← threshold, not == 1.0
        neg_mask = ~pos_mask

        pos_loss = torch.pow(1.0 - pred[pos_mask], self.alpha) * \
                   torch.log(pred[pos_mask])

        neg_loss = torch.pow(1.0 - tgt[neg_mask], self.beta) * \
                   torch.pow(pred[neg_mask], self.alpha) * \
                   torch.log(1.0 - pred[neg_mask])

        n_pos = max(pos_mask.sum().item(), 1)
        loss  = -(pos_loss.sum() + neg_loss.sum()) / n_pos
        return loss


heatmap_criterion = FocalLossHeatmap(CFG["focal_alpha"], CFG["focal_beta"], pos_thresh=0.99)





def compute_refinement_loss(obj_logits, box_deltas, anchors,
                            batch_splits, gt_boxes_batch, gt_mask_batch,
                            iou_pos_thr=0.15, iou_neg_thr=0.05):
    bce = nn.BCEWithLogitsLoss(reduction="none")
    obj_losses, box_losses = [], []

    ptr = 0
    B   = len(batch_splits)
    for b in range(B):
        N   = batch_splits[b]
        anc = anchors[b]
        obj = obj_logits[ptr:ptr + N]
        box = box_deltas[ptr:ptr + N]
        ptr += N

        gt    = gt_boxes_batch[b]
        msk   = gt_mask_batch[b]
        valid_gt = gt[msk > 0]

        if valid_gt.shape[0] == 0:
            tgt_obj = torch.zeros(N, 1, device=obj.device)
            obj_losses.append(bce(obj, tgt_obj).mean())
            continue

        anc_xyxy = torch.stack([
            anc[:, 0] - anc[:, 2] / 2,
            anc[:, 1] - anc[:, 3] / 2,
            anc[:, 0] + anc[:, 2] / 2,
            anc[:, 1] + anc[:, 3] / 2,
        ], dim=1)

        iou_mat       = tv_ops.box_iou(anc_xyxy, valid_gt.to(anc_xyxy.device))
        best_iou, best_gt_idx = iou_mat.max(dim=1)

        pos_mask = best_iou >= iou_pos_thr
        neg_mask = best_iou <  iou_neg_thr

        # Guarantee at least one positive per GT object
        for g in range(valid_gt.shape[0]):
            best_anc = iou_mat[:, g].argmax()
            pos_mask[best_anc] = True

        tgt_obj = torch.zeros(N, 1, device=obj.device)
        tgt_obj[pos_mask] = 1.0

        weight = torch.ones(N, 1, device=obj.device)
        weight[~pos_mask & ~neg_mask] = 0.0   # ignore ambiguous

        obj_losses.append(
            (bce(obj, tgt_obj) * weight).sum() /
            weight.sum().clamp(min=1))

        if pos_mask.sum() > 0:
            matched_gt  = valid_gt[best_gt_idx[pos_mask]]
            p_anc       = anc[pos_mask]

            gt_cx = (matched_gt[:, 0] + matched_gt[:, 2]) / 2
            gt_cy = (matched_gt[:, 1] + matched_gt[:, 3]) / 2
            gt_w  = (matched_gt[:, 2] - matched_gt[:, 0]).clamp(min=1)
            gt_h  = (matched_gt[:, 3] - matched_gt[:, 1]).clamp(min=1)

            tgt_dx = (gt_cx - p_anc[:, 0]) / p_anc[:, 2].clamp(min=1)
            tgt_dy = (gt_cy - p_anc[:, 1]) / p_anc[:, 3].clamp(min=1)
            tgt_dw = torch.log(gt_w / p_anc[:, 2].clamp(min=1))
            tgt_dh = torch.log(gt_h / p_anc[:, 3].clamp(min=1))
            tgt_box = torch.stack([tgt_dx, tgt_dy, tgt_dw, tgt_dh], dim=1)

            box_losses.append(
                F.l1_loss(box[pos_mask],
                          tgt_box.to(box.device)))

    obj_loss = torch.stack(obj_losses).mean() if obj_losses else \
               torch.tensor(0., device=obj_logits.device)
    box_loss = torch.stack(box_losses).mean() if box_losses else \
               torch.tensor(0., device=obj_logits.device)
    return obj_loss, box_loss


print("Loss functions defined.")

Loss functions defined.


In [13]:
# =============================================================================
# CELL 13 — Optimiser and LR scheduler (rebuilt for corrected model)
# =============================================================================

def build_optimizer_scheduler(model, cfg, steps_per_epoch):
    backbone_params = list(model.backbone.parameters()) + \
                      list(model.temporal_fuse.parameters())
    head_params     = [p for n, p in model.named_parameters()
                       if not any(n.startswith(k)
                                  for k in ["backbone", "temporal_fuse"])]

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg["lr"] * 0.1},
        {"params": head_params,     "lr": cfg["lr"]},
    ], weight_decay=cfg["weight_decay"])

    total_steps  = cfg["epochs"] * steps_per_epoch
    warmup_steps = cfg["warmup_epochs"] * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return (cfg["cosine_min_lr"] / cfg["lr"] +
                0.5 * (1 - cfg["cosine_min_lr"] / cfg["lr"]) *
                (1 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


eff_steps = len(train_dl) // CFG["accum_steps"]
optimizer, scheduler = build_optimizer_scheduler(model, CFG, eff_steps)
scaler = GradScaler(enabled=CFG["amp"])
print("Optimiser and scheduler ready.")

Optimiser and scheduler ready.


In [14]:
# =============================================================================
# CELL 14 — Decode predicted deltas → absolute xyxy boxes
# =============================================================================

def decode_boxes(box_deltas, anchors):
    """
    box_deltas : N×4  (dx, dy, dw, dh)
    anchors    : N×4  (cx, cy, pw, ph) in image pixel space
    Returns    : N×4  (x1, y1, x2, y2)
    """
    cx = anchors[:, 0] + box_deltas[:, 0] * anchors[:, 2]
    cy = anchors[:, 1] + box_deltas[:, 1] * anchors[:, 3]
    w  = anchors[:, 2] * torch.exp(box_deltas[:, 2].clamp(-4, 4))
    h  = anchors[:, 3] * torch.exp(box_deltas[:, 3].clamp(-4, 4))
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2
    return torch.stack([x1, y1, x2, y2], dim=1)


def decode_predictions(obj_logits, box_deltas, anchors,
                       batch_splits, conf_thr=0.3, nms_iou_thr=0.3):
    """
    anchors     : list (B) of N×4  [cx,cy,pw,ph]
    batch_splits: list of ints summing to total_N
    Returns list (B) of {'boxes': N×4 xyxy cpu, 'scores': N cpu}
    """
    results = []
    ptr = 0
    for b, N in enumerate(batch_splits):
        obj   = torch.sigmoid(obj_logits[ptr:ptr + N, 0])   # N
        delta = box_deltas[ptr:ptr + N]                      # N×4
        anc_b = anchors[b].to(delta.device)                  # N×4

        boxes = decode_boxes(delta, anc_b)                   # N×4 xyxy

        keep  = obj >= conf_thr
        boxes = boxes[keep]
        obj   = obj[keep]

        if boxes.shape[0] > 0:
            nms_k = tv_ops.nms(boxes, obj, nms_iou_thr)
            boxes = boxes[nms_k]
            obj   = obj[nms_k]

        results.append({"boxes": boxes.cpu(), "scores": obj.cpu()})
        ptr += N
    return results


print("Decoder defined.")

Decoder defined.


In [15]:
# =============================================================================
# CELL 15 — Evaluation metrics
#            AP@0.5, AP@0.25, AP@0.5:0.95, AR@1, AR@10, F1@0.5
#            + FPS measurement
# =============================================================================

class COCOEvaluator:
    def __init__(self, coco_gt_path):
        self.coco_gt   = COCO(coco_gt_path)
        self.results   = []   # list of COCO-format prediction dicts
        self.img_ids   = []

    def reset(self):
        self.results = []
        self.img_ids = []

    def update(self, predictions, img_ids):
        """
        predictions : list (B) of {'boxes': N×4 xyxy, 'scores': N}
        img_ids     : list of int
        """
        for pred, img_id in zip(predictions, img_ids):
            self.img_ids.append(img_id)
            boxes  = pred["boxes"]
            scores = pred["scores"]
            for i in range(boxes.shape[0]):
                x1, y1, x2, y2 = boxes[i].tolist()
                w, h = x2 - x1, y2 - y1
                self.results.append({
                    "image_id"   : int(img_id),
                    "category_id": 1,
                    "bbox"       : [x1, y1, w, h],
                    "score"      : float(scores[i]),
                })

    def evaluate(self):
        if not self.results:
            return {k: 0.0 for k in
                    ["AP50", "AP25", "AP50_95", "AR1", "AR10", "F1_50"]}

        coco_dt = self.coco_gt.loadRes(self.results)

        # ── AP@0.5:0.95 ──────────────────────────────────────────────────
        ev = COCOeval(self.coco_gt, coco_dt, "bbox")
        ev.params.iouThrs = np.linspace(0.5, 0.95, 10)
        ev.evaluate(); ev.accumulate(); ev.summarize()
        ap_50_95 = float(ev.stats[0])
        ar_10    = float(ev.stats[9])   # AR@10
        ar_1     = float(ev.stats[6])   # AR@1

        # ── AP@0.5 ───────────────────────────────────────────────────────
        ev50 = COCOeval(self.coco_gt, coco_dt, "bbox")
        ev50.params.iouThrs = np.array([0.5])
        ev50.evaluate(); ev50.accumulate(); ev50.summarize()
        ap_50 = float(ev50.stats[0])
        # precision/recall at 0.5 for F1
        # stats[0]=AP, stats[8]=AR@maxDets=10 at single threshold
        # Use raw precision-recall curve
        precision = ev50.eval["precision"][0, :, 0, 0, 2]  # [iou,recall,cat,area,maxdet]
        recall    = ev50.params.recThrs
        f1_vals   = 2 * precision * recall / (precision + recall + 1e-8)
        f1_50     = float(f1_vals.max()) if len(f1_vals) > 0 else 0.0

        # ── AP@0.25 ───────────────────────────────────────────────────────
        ev25 = COCOeval(self.coco_gt, coco_dt, "bbox")
        ev25.params.iouThrs = np.array([0.25])
        ev25.evaluate(); ev25.accumulate(); ev25.summarize()
        ap_25 = float(ev25.stats[0])

        return {
            "AP50"    : ap_50,
            "AP25"    : ap_25,
            "AP50_95" : ap_50_95,
            "AR1"     : ar_1,
            "AR10"    : ar_10,
            "F1_50"   : f1_50,
        }


def measure_fps(model, device, img_size=640, T=3, n_runs=50, warmup=10):
    model.eval()
    dummy = torch.randn(1, T, 3, img_size, img_size, device=device)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_runs):
            _ = model(dummy)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
    fps = n_runs / (t1 - t0)
    model.train()
    return fps


val_evaluator  = COCOEvaluator(VAL_COCO_JSON)
test_evaluator = COCOEvaluator(TEST_COCO_JSON)
print("Metrics engine ready.")

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.07s)
creating index...
index created!
Metrics engine ready.


In [16]:
# =============================================================================
# CELL 16 — Train one epoch / validate one epoch
# =============================================================================

def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    total_loss   = 0.0
    total_hm     = 0.0
    total_obj    = 0.0
    total_box    = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        frames    = batch["frames"].to(DEVICE)           # B×T×3×H×W
        hm_target = batch["heatmap"].to(DEVICE)          # B×1×HM_H×HM_W
        gt_boxes  = batch["gt_boxes"].to(DEVICE)         # B×max_objs×4
        gt_mask   = batch["gt_mask"].to(DEVICE)          # B×max_objs

        with autocast(enabled=CFG["amp"]):
            hm_logits, obj_logits, box_deltas, anchors, splits = model(frames)

            # ── heatmap loss ─────────────────────────────────────────────
            
            lh = heatmap_criterion(hm_logits, hm_target)

            # ── refinement losses ────────────────────────────────────────
            lo, lb = compute_refinement_loss(
                obj_logits, box_deltas, anchors, splits, gt_boxes, gt_mask)

            loss = (CFG["heatmap_loss_w"] * lh +
                    CFG["obj_loss_w"]     * lo +
                    CFG["box_loss_w"]     * lb)
            loss = loss / CFG["accum_steps"]

        scaler.scale(loss).backward()

        if (step + 1) % CFG["accum_steps"] == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["clip_grad"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG["accum_steps"]
        total_hm   += lh.item()
        total_obj  += lo.item()
        total_box  += lb.item()

        if step % 50 == 0:
            lr_now = scheduler.get_last_lr()[0]
            print(f"  [E{epoch}|{step}/{len(loader)}] "
                  f"loss={total_loss/(step+1):.4f}  "
                  f"hm={total_hm/(step+1):.4f}  "
                  f"obj={total_obj/(step+1):.4f}  "
                  f"box={total_box/(step+1):.4f}  "
                  f"lr={lr_now:.2e}")

    n = len(loader)
    return {
        "train/loss"    : total_loss / n,
        "train/hm_loss" : total_hm   / n,
        "train/obj_loss": total_obj  / n,
        "train/box_loss": total_box  / n,
    }


@torch.no_grad()
def validate(model, loader, evaluator, conf_thr, nms_iou_thr):
    model.eval()
    evaluator.reset()
    total_loss = 0.0

    for batch in loader:
        frames    = batch["frames"].to(DEVICE)
        hm_target = batch["heatmap"].to(DEVICE)
        gt_boxes  = batch["gt_boxes"].to(DEVICE)
        gt_mask   = batch["gt_mask"].to(DEVICE)
        img_ids   = batch["img_id"].tolist()

        with autocast(enabled=CFG["amp"]):
            hm_logits, obj_logits, box_deltas, anchors, splits = model(frames)
            lh       = heatmap_criterion(hm_logits, hm_target)
            lo, lb   = compute_refinement_loss(
                obj_logits, box_deltas, anchors, splits, gt_boxes, gt_mask)
            loss = (CFG["heatmap_loss_w"] * lh +
                    CFG["obj_loss_w"]     * lo +
                    CFG["box_loss_w"]     * lb)
        total_loss += loss.item()

        preds = decode_predictions(
            obj_logits, box_deltas, anchors, splits,
            conf_thr=conf_thr, nms_iou_thr=nms_iou_thr)
        evaluator.update(preds, img_ids)

    metrics  = evaluator.evaluate()
    val_loss = total_loss / len(loader)
    return val_loss, metrics


print("Train/val functions defined.")

Train/val functions defined.


In [17]:
# =============================================================================
# CELL 17 — Checkpoint save/load utilities
# =============================================================================

def save_checkpoint(model, optimizer, scheduler, epoch, metrics, tag="last"):
    ckpt = {
        "epoch"    : epoch,
        "model"    : model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "metrics"  : metrics,
        "cfg"      : CFG,
    }
    path = os.path.join(CFG["save_dir"], f"checkpoint_{tag}.pt")
    torch.save(ckpt, path)
    return path


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    if optimizer  and "optimizer"  in ckpt: optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler  and "scheduler"  in ckpt: scheduler.load_state_dict(ckpt["scheduler"])
    return ckpt.get("epoch", 0), ckpt.get("metrics", {})


def log_checkpoint_to_wandb(path, tag):
    artifact = wandb.Artifact(
        name=f"{CFG['run_name']}-{tag}",
        type="model",
        description=f"Checkpoint: {tag}",
    )
    artifact.add_file(path)
    wandb.log_artifact(artifact)
    print(f"  → W&B artifact logged: {tag}")


print("Checkpoint utilities ready.")

Checkpoint utilities ready.


In [20]:
# =============================================================================
# CELL 17.8 — Pull checkpoint_epoch_014 back from W&B (same run, new session)
# =============================================================================

import wandb as _wandb_check
import shutil

assert wandb.run is not None, "wandb.init() must be called before this cell."

ARTIFACT_TAG = "epoch_014"
ARTIFACT_NAME = f"{CFG['run_name']}-{ARTIFACT_TAG}"

artifact_ref = f"{wandb.run.entity}/{CFG['project']}/{ARTIFACT_NAME}:latest"

print(f"Pulling artifact: {artifact_ref}")
api = wandb.Api()
artifact = api.artifact(artifact_ref, type="model")
artifact_dir = artifact.download()

print(f"Downloaded to: {artifact_dir}")
print(f"Contents: {os.listdir(artifact_dir)}")

downloaded_files = [f for f in os.listdir(artifact_dir) if f.endswith(".pt")]
assert len(downloaded_files) == 1, (
    f"Expected exactly 1 .pt file in artifact, found: {downloaded_files}"
)

src_path = os.path.join(artifact_dir, downloaded_files[0])
dst_path = os.path.join(CFG["save_dir"], "checkpoint_epoch_014.pt")

os.makedirs(CFG["save_dir"], exist_ok=True)
shutil.copy(src_path, dst_path)

print(f"Checkpoint restored to: {dst_path}")

# Quick integrity check before resume
ckpt_check = torch.load(dst_path, map_location="cpu")
print(f"Checkpoint epoch field : {ckpt_check.get('epoch')}")
print(f"Checkpoint AP50        : {ckpt_check.get('metrics', {}).get('AP50')}")

assert ckpt_check.get("epoch") == 14, (
    f"Loaded checkpoint reports epoch={ckpt_check.get('epoch')}, expected 14. "
    f"Wrong artifact pulled — check ARTIFACT_TAG and project/entity."
)

print("✅ Checkpoint verified, ready for resume.")

Pulling artifact: bscs22030-information-technology-university/drone-detection-nps/heatmap-guided-sparse-detector-v1-epoch_014:latest


wandb: Downloading large artifact 'heatmap-guided-sparse-detector-v1-epoch_014:latest', 74.86MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (458.3MB/s)


Downloaded to: /kaggle/working/artifacts/heatmap-guided-sparse-detector-v1-epoch_014:v0
Contents: ['checkpoint_epoch_014.pt']
Checkpoint restored to: /kaggle/working/checkpoints/checkpoint_epoch_014.pt
Checkpoint epoch field : 14
Checkpoint AP50        : 0.7754623916379423
✅ Checkpoint verified, ready for resume.


In [ ]:
# =============================================================================
# CELL 18 — Main training loop (resume from epoch 9 checkpoint → epoch 10)
# =============================================================================

RESUME = True
RESUME_PATH = os.path.join(CFG["save_dir"], "checkpoint_epoch_014.pt")

start_epoch = 1
best_ap50   = -1.0
history     = []

if RESUME:
    if not os.path.exists(RESUME_PATH):
        _fail_msg = (f"Resume checkpoint not found at {RESUME_PATH}. "
                     f"List available checkpoints below and pick the correct path.")
        print(_fail_msg)
        print(sorted(os.listdir(CFG["save_dir"])))
        raise FileNotFoundError(_fail_msg)

    loaded_epoch, loaded_metrics = load_checkpoint(
        RESUME_PATH, model, optimizer, scheduler)

    if loaded_epoch != 9:
        print(f"⚠ WARNING: checkpoint reports epoch={loaded_epoch}, "
              f"expected 9. Verify {RESUME_PATH} is the correct file.")

    start_epoch = loaded_epoch + 1   # → 10
    best_ap50   = loaded_metrics.get("AP50", -1.0)

    print(f"Resumed from {RESUME_PATH}")
    print(f"  loaded_epoch     : {loaded_epoch}")
    print(f"  resuming at      : epoch {start_epoch}")
    print(f"  best AP50 so far : {best_ap50:.4f}")
    print(f"  optimizer LR     : {scheduler.get_last_lr()}")
else:
    print("Starting fresh training run (RESUME=False).")

for epoch in range(start_epoch, CFG["epochs"] + 1):
    t_start = time.time()
    print(f"\n{'='*60}")
    print(f" Epoch {epoch}/{CFG['epochs']}")
    print(f"{'='*60}")

    # ── train ─────────────────────────────────────────────────────────────
    train_metrics = train_one_epoch(
        model, train_dl, optimizer, scheduler, scaler, epoch)

    # ── validate ──────────────────────────────────────────────────────────
    val_loss, val_metrics = validate(
        model, val_dl, val_evaluator,
        conf_thr    = CFG["conf_thr"],
        nms_iou_thr = CFG["nms_iou_thr"])

    # ── FPS (every 5 epochs) ──────────────────────────────────────────────
    fps = measure_fps(model, DEVICE) if epoch % 5 == 0 else None

    elapsed = time.time() - t_start

    # ── assemble log dict ─────────────────────────────────────────────────
    log = {
        **train_metrics,
        "val/loss"    : val_loss,
        "val/AP50"    : val_metrics["AP50"],
        "val/AP25"    : val_metrics["AP25"],
        "val/AP50_95" : val_metrics["AP50_95"],
        "val/AR1"     : val_metrics["AR1"],
        "val/AR10"    : val_metrics["AR10"],
        "val/F1_50"   : val_metrics["F1_50"],
        "epoch"       : epoch,
        "epoch_time_s": elapsed,
        "lr"          : scheduler.get_last_lr()[0],
    }
    if fps is not None:
        log["fps"] = fps

    wandb.log(log)
    history.append(log)

    print(f"  val_loss={val_loss:.4f} | AP50={val_metrics['AP50']:.4f} "
          f"| AP50_95={val_metrics['AP50_95']:.4f} "
          f"| F1={val_metrics['F1_50']:.4f} "
          f"| time={elapsed:.1f}s"
          + (f" | FPS={fps:.1f}" if fps else ""))

    # ── save LAST checkpoint every epoch ─────────────────────────────────
    last_path = save_checkpoint(
        model, optimizer, scheduler, epoch, val_metrics, tag="last")
    log_checkpoint_to_wandb(last_path, tag="last")

    # ── save EPOCH checkpoint ─────────────────────────────────────────────
    epoch_path = save_checkpoint(
        model, optimizer, scheduler, epoch, val_metrics,
        tag=f"epoch_{epoch:03d}")
    log_checkpoint_to_wandb(epoch_path, tag=f"epoch_{epoch:03d}")

    # ── save BEST checkpoint ───────────────────────────────────────────────
    if val_metrics["AP50"] > best_ap50:
        best_ap50 = val_metrics["AP50"]
        best_path = save_checkpoint(
            model, optimizer, scheduler, epoch, val_metrics, tag="best")
        log_checkpoint_to_wandb(best_path, tag="best")
        print(f"  ★ New best AP50={best_ap50:.4f} → saved best checkpoint")
        wandb.run.summary["best_AP50"]   = best_ap50
        wandb.run.summary["best_epoch"]  = epoch


print("\nTraining complete.")

⚠ WARNING: checkpoint reports epoch=14, expected 9. Verify /kaggle/working/checkpoints/checkpoint_epoch_014.pt is the correct file.
Resumed from /kaggle/working/checkpoints/checkpoint_epoch_014.pt
  loaded_epoch     : 14
  resuming at      : epoch 15
  best AP50 so far : 0.7755
  optimizer LR     : [1.8713923467341786e-05, 0.00018713923467341786]

 Epoch 15/60
  [E15|0/8037] loss=3.9855  hm=1.3238  obj=0.0241  box=1.3188  lr=1.87e-05
  [E15|50/8037] loss=2.5474  hm=0.9775  obj=0.0278  box=0.7710  lr=1.87e-05
  [E15|100/8037] loss=2.3161  hm=0.9567  obj=0.0272  box=0.6661  lr=1.87e-05
  [E15|150/8037] loss=2.1446  hm=0.8825  obj=0.0257  box=0.6182  lr=1.87e-05
  [E15|200/8037] loss=2.1683  hm=0.9040  obj=0.0267  box=0.6188  lr=1.87e-05
  [E15|250/8037] loss=2.0352  hm=0.8758  obj=0.0263  box=0.5665  lr=1.87e-05
  [E15|300/8037] loss=2.0853  hm=0.8840  obj=0.0262  box=0.5875  lr=1.87e-05
  [E15|350/8037] loss=2.1254  hm=0.8864  obj=0.0264  box=0.6063  lr=1.87e-05
  [E15|400/8037] loss=2.

In [18]:
# =============================================================================
# CELL 19 — Download best artifact from W&B and run Test evaluation
# =============================================================================
import os
import torch
import wandb

# 1. Define the artifact name path using your configuration
# Format: '<entity>/<project>/<artifact_name>:<version_or_tag>'
# If entity/project aren't specified, wandb uses the current active run's context.
artifact_path = f"{CFG['run_name']}-best:latest"

print(f"Downloading best checkpoint artifact from W&B: {artifact_path}...")
try:
    # Use the wandb API to download the artifact
    artifact = wandb.use_artifact(artifact_path, type="model")
    artifact_dir = artifact.download()
    
    # The downloaded file will keep its original filename 'checkpoint_best.pt' inside artifact_dir
    best_ckpt = os.path.join(artifact_dir, "checkpoint_best.pt")
    print(f"Artifact downloaded successfully to: {best_ckpt}")
    
except Exception as e:
    print(f"Failed to fetch from W&B artifact registry: {e}")
    # Fallback locally just in case it exists in the output directory
    best_ckpt = os.path.join(CFG["save_dir"], "checkpoint_best.pt")
    print(f"Attempting local fallback path: {best_ckpt}")

# 2. Load the downloaded checkpoint into your model
load_checkpoint(best_ckpt, model)
print(f"Loaded best checkpoint into model.")

# 3. Measure FPS on test hardware
fps_test = measure_fps(model, DEVICE, n_runs=100, warmup=20)
print(f"FPS (batch=1, T=3, 640×640): {fps_test:.2f}")

# 4. Evaluate on the test dataset
test_loss, test_metrics = validate(
    model, test_dl, test_evaluator,
    conf_thr    = CFG["conf_thr"],
    nms_iou_thr = CFG["nms_iou_thr"])

print("\n" + "="*50)
print(" TEST RESULTS")
print("="*50)
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v:.4f}")
print(f"  {'FPS':12s}: {fps_test:.2f}")
print("="*50)

# 5. Log the performance profile back to W&B
test_log = {
    "test/loss"    : test_loss,
    "test/AP50"    : test_metrics["AP50"],
    "test/AP25"    : test_metrics["AP25"],
    "test/AP50_95" : test_metrics["AP50_95"],
    "test/AR1"     : test_metrics["AR1"],
    "test/AR10"    : test_metrics["AR10"],
    "test/F1_50"   : test_metrics["F1_50"],
    "test/FPS"     : fps_test,
}
wandb.log(test_log)

for k, v in test_log.items():
    wandb.run.summary[k] = v

wandb: Downloading large artifact 'heatmap-guided-sparse-detector-v1-best:latest', 74.86MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:01.7 (45.3MB/s)


Artifact downloaded successfully to: /kaggle/working/artifacts/heatmap-guided-sparse-detector-v1-best:v12/checkpoint_best.pt
Loaded best checkpoint into model.
FPS (batch=1, T=3, 640×640): 34.15
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.21s).
Accumulating evaluation results...
DONE (t=0.46s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.228
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.649
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.079
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.228
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.212
 Average Recall     (AR) @[ IoU

In [19]:
# =============================================================================
# CELL 20 — Qualitative visualisation (last frame of triplet + predictions)
# =============================================================================

mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def unnorm(t):
    return (t * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()


model.eval()
vis_images = []
n_vis = min(8, len(test_triplets))

with torch.no_grad():
    for idx in range(n_vis):
        batch = test_dl.dataset[idx]
        frames    = batch["frames"].unsqueeze(0).to(DEVICE)
        gt_boxes  = batch["gt_boxes"]
        gt_mask   = batch["gt_mask"]

        hm_logits, obj_logits, box_deltas, anchors, splits = model(frames)
        preds = decode_predictions(obj_logits, box_deltas, anchors, splits,
                                   conf_thr=CFG["conf_thr"],
                                   nms_iou_thr=CFG["nms_iou_thr"])

        anchor_frame = unnorm(batch["frames"][-1])  # last frame
        img_vis = (anchor_frame * 255).astype(np.uint8).copy()
        H, W, _ = img_vis.shape

        # Draw GT (green)
        for i in range(int(gt_mask.sum())):
            x1, y1, x2, y2 = gt_boxes[i].tolist()
            cv2.rectangle(img_vis,
                          (int(x1), int(y1)), (int(x2), int(y2)),
                          (0, 255, 0), 1)

        # Draw preds (red)
        pred = preds[0]
        for i in range(pred["boxes"].shape[0]):
            x1, y1, x2, y2 = pred["boxes"][i].tolist()
            sc = pred["scores"][i].item()
            cv2.rectangle(img_vis,
                          (int(x1), int(y1)), (int(x2), int(y2)),
                          (255, 0, 0), 1)
            cv2.putText(img_vis, f"{sc:.2f}",
                        (int(x1), max(0, int(y1) - 3)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 0, 0), 1)

        vis_images.append(img_vis)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, ax in enumerate(axes.flat):
    if i < len(vis_images):
        ax.imshow(vis_images[i])
        ax.set_title(f"Sample {i}  🟩GT  🟥Pred")
    ax.axis("off")
plt.suptitle("Test Set Qualitative Results", fontsize=14)
plt.tight_layout()
wandb.log({"test/qualitative": wandb.Image(fig)})
plt.show()

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [ ]:
# =============================================================================
# CELL 21 — Training curves
# =============================================================================

df = pd.DataFrame(history)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].plot(df["epoch"], df["train/loss"], label="Train Loss")
axes[0,0].plot(df["epoch"], df["val/loss"],   label="Val Loss")
axes[0,0].set_title("Loss"); axes[0,0].legend()

axes[0,1].plot(df["epoch"], df["val/AP50"],    label="AP@0.5")
axes[0,1].plot(df["epoch"], df["val/AP25"],    label="AP@0.25")
axes[0,1].plot(df["epoch"], df["val/AP50_95"], label="AP@0.5:0.95")
axes[0,1].set_title("AP Metrics"); axes[0,1].legend()

axes[0,2].plot(df["epoch"], df["val/F1_50"], label="F1@0.5", color="purple")
axes[0,2].set_title("F1@0.5"); axes[0,2].legend()

axes[1,0].plot(df["epoch"], df["val/AR1"],  label="AR@1")
axes[1,0].plot(df["epoch"], df["val/AR10"], label="AR@10")
axes[1,0].set_title("Average Recall"); axes[1,0].legend()

axes[1,1].plot(df["epoch"], df["train/hm_loss"],  label="Heatmap")
axes[1,1].plot(df["epoch"], df["train/obj_loss"],  label="Objectness")
axes[1,1].plot(df["epoch"], df["train/box_loss"],  label="Box")
axes[1,1].set_title("Component Losses"); axes[1,1].legend()

axes[1,2].plot(df["epoch"], df["lr"], label="LR", color="orange")
axes[1,2].set_title("Learning Rate"); axes[1,2].legend()

plt.suptitle("Training History", fontsize=14)
plt.tight_layout()
wandb.log({"training_curves": wandb.Image(fig)})
plt.show()

df.to_csv("/kaggle/working/training_history.csv", index=False)
print("Training history saved.")

In [ ]:
# =============================================================================
# CELL 22 — Heatmap activation visualisation (qualitative inspection)
# =============================================================================

model.eval()
batch = next(iter(val_dl))
frames    = batch["frames"][:4].to(DEVICE)
hm_target = batch["heatmap"][:4]

with torch.no_grad():
    hm_logits, *_ = model(frames)
hm_pred = torch.sigmoid(hm_logits[:4]).cpu()

fig, axes = plt.subplots(4, 2, figsize=(10, 20))
for i in range(4):
    img = unnorm(batch["frames"][i, -1])
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Frame {i}"); axes[i, 0].axis("off")
    axes[i, 1].imshow(hm_pred[i, 0].numpy(), cmap="inferno", vmin=0, vmax=1)
    axes[i, 1].set_title(f"Pred Heatmap {i}"); axes[i, 1].axis("off")

plt.suptitle("Predicted Heatmaps (val)", fontsize=13)
plt.tight_layout()
wandb.log({"heatmap_activations": wandb.Image(fig)})
plt.show()

In [ ]:
# =============================================================================
# CELL 23 — Finalise W&B run
# =============================================================================

wandb.run.summary["final_best_AP50"] = best_ap50
wandb.finish()
print("W&B run finished. All done.")